# 201 · Self-describing versus schema-dependent formats

This notebook goes with the article
[Self-describing vs schema](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/self-describing-vs-schema-dependent/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/self_describing_vs_schema.ipynb)

The deep design question is not only “text or binary?”
It is **where the meaning of each field lives** when the message is in transit:
in the payload itself (names and type tags), or in a shared contract both sides already agree on.

You will encode one small record several ways and compare size and whether field names appear in the bytes.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


In [ ]:
import json

RECORD = {"user_id": 42, "name": "Ada", "balance": 100}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("pip install msgpack  # optional for MessagePack cells")



## JSON: names on every message

JSON is the textbook self-describing case: field names and structure travel with the data.
After encoding, you can find the key strings as UTF-8 inside the byte sequence.

That makes debugging easy. The trade-off is that every message repeats that metadata.


In [ ]:
def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)

j = json.dumps(RECORD, separators=(",", ":")).encode("utf-8")
print(j.decode())
print("nbytes", len(j))
print("hex", hex_bytes(j)[:80], "…")
# Field names are UTF-8 substrings on the wire:
for key in RECORD:
    assert key.encode() in j
print("OK: each key appears as UTF-8 in the payload")



## MessagePack: still a dynamic model

MessagePack is binary, but it is not the same as a schema-driven Protocol Buffers message.
When you pack a **map**, keys still appear on the wire.
When you pack an **array** in a fixed order, the keys disappear and order becomes the contract.

So “we moved to binary” does not automatically mean “we stopped shipping field names.”


In [ ]:
if not HAS_MSGPACK:
    print("SKIP msgpack")
else:
    packed = msgpack.packb(RECORD, use_bin_type=True)
    print("nbytes", len(packed))
    print("hex", hex_bytes(packed))
    # keys still present for map encoding
    for key in RECORD:
        assert key.encode() in packed
    print("OK: map keys still travel with MessagePack maps")
    # array form drops names — schema-like discipline without IDL
    as_array = msgpack.packb([RECORD["user_id"], RECORD["name"], RECORD["balance"]], use_bin_type=True)
    print("as array nbytes", len(as_array), "hex", hex_bytes(as_array))
    for key in RECORD:
        assert key.encode() not in as_array
    print("OK: array form has no field names (order is the contract)")



## Schema-dependent sketch: field numbers, no names

This tiny encoder uses field numbers `1`, `2`, and `3` the way classic Protocol Buffers does on the wire.
Field names should **not** appear in the bytes; a reader needs the shared schema (or equivalent knowledge) to interpret the numbers.

The sketch is for teaching only. Production systems generate code from a `.proto` file (or similar) instead of hand-rolling every field.


In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_record_pb_style(r: dict) -> bytes:
    # 1:user_id varint, 2:name string, 3:balance as fixed64 bits of float64 (toy)
    import struct
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(int(r["user_id"]))
    name = r["name"].encode()
    out += encode_key(2, 2) + encode_varint(len(name)) + name
    out += encode_key(3, 1) + struct.pack("<d", float(r["balance"]))  # wire type 1 = 8 bytes
    return bytes(out)


pb = encode_record_pb_style(RECORD)
print("pb-style nbytes", len(pb), "hex", hex_bytes(pb))
for key in RECORD:
    assert key.encode() not in pb
print("OK: field names absent; numbers 1/2/3 carry identity via shared schema")



## Size table for this payload only

The printed table is for **this one record**, not a universal ranking of formats.
You will usually see JSON largest, a MessagePack map smaller, and keyless forms smaller still—
but the exact order can shift with different field names and values.


In [ ]:
rows = [("JSON", len(j)), ("pb-style sketch", len(pb))]
if HAS_MSGPACK:
    rows.insert(1, ("MessagePack map", len(msgpack.packb(RECORD, use_bin_type=True))))
    rows.insert(2, ("MessagePack array", len(msgpack.packb(list(RECORD.values()), use_bin_type=True))))
for name, n in rows:
    print(f"{name:20} {n:4} bytes")



## Takeaways

- **Self-describing** formats carry much of the meaning inside each message (names and/or type tags).
- **Schema-dependent** formats keep meaning in a shared contract; the wire can stay dense and opaque by itself.
- Think of a spectrum: JSON → MessagePack map → MessagePack array → field-number encodings.

In design reviews, ask “who carries field identity?” before arguing only about milliseconds.

**Next:** [Schema evolution](./schema_evolution.ipynb)
